In [1]:
import json
import asyncio
import sys
from pathlib import Path

# Walk up from CWD to find the project root (identified by pyproject.toml),
# so imports work regardless of where Jupyter is launched from.
def find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError(f"Could not find project root from {start}")

project_root = find_project_root(Path().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from movie_ingestion.metadata_generation.inputs import MovieInputData
from movie_ingestion.metadata_generation.generators.plot_events import generate_plot_events

# Load all movies from the saved JSON
with open(project_root / "saved_imdb_movies.json") as f:
    raw_movies = json.load(f)

# Convert raw dicts to MovieInputData objects.
# The JSON uses debug_synopses/debug_plot_summaries (field names used during
# scraping) and release_date ("YYYY-MM-DD") instead of release_year.
def to_movie_input(raw: dict) -> MovieInputData:
    release_date = raw.get("release_date") or ""
    release_year = int(release_date[:4]) if release_date else None
    return MovieInputData(
        tmdb_id=raw["tmdb_id"],
        title=raw["title"],
        release_year=release_year,
        overview=raw.get("overview", ""),
        genres=raw.get("genres", []),
        plot_synopses=raw.get("debug_synopses", []),
        plot_summaries=raw.get("debug_plot_summaries", []),
        plot_keywords=raw.get("plot_keywords", []),
        overall_keywords=raw.get("overall_keywords", []),
        featured_reviews=raw.get("featured_reviews", []),
        reception_summary=raw.get("reception_summary"),
        audience_reception_attributes=raw.get("audience_reception_attributes", []),
        maturity_rating=raw.get("maturity_rating", ""),
        maturity_reasoning=raw.get("maturity_reasoning", []),
        parental_guide_items=raw.get("parental_guide_items", []),
    )

movies = [to_movie_input(r) for r in raw_movies]
print(f"Loaded {len(movies)} movies")

# Run generate_plot_events on the first movie
first_movie = movies[0]
print(f"Running generate_plot_events for: {first_movie.title_with_year()}")

result, token_usage = await generate_plot_events(first_movie)
print(f"\nResult:\n{result}")
print(f"\nToken usage: {token_usage}")


/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 50 movies
Running generate_plot_events for: ferris bueller's day off (1986)

Result:
high school senior ferris bueller fakes illness at home in suburban chicago by setting up a mannequin in bed and staging a sickroom while his parents (tom and katie bueller) believe he's sick. he persuades pessimistic best friend cameron frye to lend his father's restored 1961 ferrari 250 gt california and enlists girlfriend sloane peterson to join a day in downtown chicago. ferris has cameron impersonate sloane's father by phone so sloane can be excused for a fake family death. dean of students edward rooney, convinced ferris is truant, leaves school to hunt him down. ferris, sloane, and cameron park the ferrari with garage attendants who shortly joyride the car. the trio tour chicago: wrigley field, the sears tower, the art institute of chicago (where ferris gives cameron a moment of calm), the chicago mercantile exchange, and they join the von steuben day parade where ferris lip-syncs to wayn